In [ ]:
# =========================
# Standard library imports
# =========================
import sys
import os
import io
import gzip
from pathlib import Path
import math
import json
import pickle
import logging
import random
import subprocess
import warnings
import collections
import itertools
import re
from IPython.display import display

warnings.filterwarnings("ignore")


# =========================
# Numeric / stats
# =========================
import numpy as np
import pandas as pd
from scipy import sparse
from scipy import stats
from scipy.stats import rankdata
from scipy import __version__ as scipy_version


# =========================
# Plotting / visualization
# =========================
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, FormatStrFormatter, MaxNLocator
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.gridspec import GridSpec
import seaborn as sns


# =========================
# scikit-learn
# =========================
from sklearn import model_selection, metrics
from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit,
    KFold,
    GroupKFold,
    cross_val_score,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.linear_model import LassoCV
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neighbors import LocalOutlierFactor
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    r2_score,
    mean_squared_error,
    mean_absolute_error,
)
from sklearn import __version__ as sklearn_version


# =========================
# Optimization / AutoML
# =========================
import optuna


# =========================
# LightGBM
# =========================
try:
    import lightgbm as lgb
    from lightgbm import LGBMRegressor
    _HAS_LGBM = True
except Exception as e:
    _HAS_LGBM = False
    raise RuntimeError(
        "LightGBM not installed. Please install it with `pip install lightgbm`."
    ) from e


# =========================
# RNA structure (ViennaRNA)
# =========================
import RNA


# =========================
# UpSet plots
# =========================
try:
    from upsetplot import UpSet, from_memberships
except Exception:
    _ = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "upsetplot"],
        check=False,
    )
    from upsetplot import UpSet, from_memberships


# =========================
# Model explanation
# =========================
import shap


# =========================
# User-provided utilities
# =========================
import sylib  


# =========================
# Version info
# =========================
print(f"python    = {sys.version_info[0]}.{sys.version_info[1]}.{sys.version_info[2]}")
print(f"pandas    = {pd.__version__}")
print(f"numpy     = {np.__version__}")
print(f"scipy     = {scipy_version}")
print(f"optuna    = {optuna.__version__}")
print(f"sklearn   = {sklearn_version}")
print(f"ViennaRNA = {RNA.__version__}")
print(f"lightgbm  = {lgb.__version__}")
print(f"sylib     = {sylib.__version__}")


# =========================
# Progress bar & logging
# =========================
# progress bar from sylib
progress_bar = sylib.utils.ProgressBar()

# reset handlers then configure logging
logging.root.handlers = []
stream_handler = logging.StreamHandler(sys.stderr)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)8s: %(message)s",
    handlers=[stream_handler],
)
logger = logging.getLogger(__name__)

# make matplotlib quieter
logging.getLogger("matplotlib").setLevel(logging.WARNING)

In [ ]:
# ============================================
# Global plotting configuration
# ============================================

_PLOT_CFG = {
    "fig_w": 6.0,
    "fig_h": 6.0,
    "dpi": 300,
}


SPECIES_INFO = {
    "AT21": {
        "label": "AT",
        "short": "AT21",
        "color": "#664D0AFF",
        "marker": "o",
    },
    "NB21": {
        "label": "NB",
        "short": "NB21",
        "color": "#7e3131",
        "marker": "^",
    },
    "OS21": {
        "label": "OS",
        "short": "OS21",
        "color": "#13563f",
        "marker": "s",
    },
}

def set_plot_style(
    *,
    base_fontsize=10,
    title_fontsize=12,
    label_fontsize=10,
    tick_fontsize=9,
    legend_fontsize=10,
    dpi=300,
    axes_linewidth=1.2,
    spines_top=True,
    spines_right=True,
    tick_size_major=6,
    tick_dir="out",
    grid=False,
    fig_w=6.0,
    fig_h=6.0,
):
    sns.set_style("ticks")

    mpl.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": base_fontsize,

        "axes.titlesize": title_fontsize,
        "axes.labelsize": label_fontsize,

        "xtick.labelsize": tick_fontsize,
        "ytick.labelsize": tick_fontsize,

        "legend.fontsize": legend_fontsize,

        "figure.dpi": dpi,
        "savefig.dpi": dpi,

        "axes.linewidth": axes_linewidth,
        "axes.spines.top": spines_top,
        "axes.spines.right": spines_right,
        "axes.grid": grid,
        "axes.axisbelow": True,

        "xtick.major.size": tick_size_major,
        "ytick.major.size": tick_size_major,
        "xtick.direction": tick_dir,
        "ytick.direction": tick_dir,

        "legend.frameon": False,

        "savefig.bbox": "tight",
        "savefig.transparent": False,
        "figure.autolayout": False,
    })

    _PLOT_CFG.update({
        "fig_w": fig_w,
        "fig_h": fig_h,
        "dpi": dpi,
    })


def make_fig(w=None, h=None, dpi=None):
    W = float(w) if w is not None else _PLOT_CFG["fig_w"]
    H = float(h) if h is not None else _PLOT_CFG["fig_h"]
    D = dpi if dpi is not None else _PLOT_CFG["dpi"]

    fig, ax = plt.subplots(
        figsize=(W, H),
        dpi=D,
    )

    return fig, ax


def _compact_formatter():
    def _fmt(x, _pos=None):
        axx = abs(x)

        if axx >= 1e9:
            s = f"{x / 1e9:.1f}B"
        elif axx >= 1e6:
            s = f"{x / 1e6:.1f}M"
        elif axx >= 1e3:
            s = f"{x / 1e3:.1f}k"
        else:
            s = f"{x:.2g}"

        return (
            s.replace(".0B", "B")
             .replace(".0M", "M")
             .replace(".0k", "k")
        )

    return FuncFormatter(_fmt)


def format_axis(
    ax,
    *,
    xlabel=None,
    ylabel=None,
    compact_ticks=(),
):
    if xlabel is not None:
        ax.set_xlabel(xlabel)

    if ylabel is not None:
        ax.set_ylabel(ylabel)

    fmt = _compact_formatter()

    if "x" in compact_ticks:
        ax.xaxis.set_major_formatter(fmt)

    if "y" in compact_ticks:
        ax.yaxis.set_major_formatter(fmt)

    return ax


# ============================================
# Joint scatter with KDE marginals
# ============================================

def safe_pearsonr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    return stats.pearsonr(x, y)


def safe_spearmanr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    if len(x) < 2:
        return np.nan, np.nan

    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan, np.nan

    return stats.spearmanr(x, y)

def _kde_1d(values, lo, hi, num=256):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    grid = np.linspace(lo, hi, num)

    if len(values) < 2:
        return grid, np.zeros_like(grid)

    try:
        kde = stats.gaussian_kde(values)
        dens = kde(grid)
        dens /= dens.max() if dens.max() > 0 else 1
        return grid, dens

    except Exception:
        return grid, np.zeros_like(grid)


def joint_scatter(
    x,
    y,
    *,
    color=None,
    point_size=18,
    alpha=0.65,
    show_identity=True,
    show_regression=True,
    annotate=True,
    annotate_spearman=True,
    xlabel=None,
    ylabel=None,
    title=None,
    figsize=None,
    w=None,
    h=None,
    dpi=None,
    annotate_fontsize=14,
):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    mask = np.isfinite(x) & np.isfinite(y)
    x = x[mask]
    y = y[mask]

    n = len(x)

    if figsize is not None:
        FW, FH = figsize
    else:
        FW = float(w) if w is not None else _PLOT_CFG["fig_w"]
        FH = float(h) if h is not None else _PLOT_CFG["fig_h"]

    fig = plt.figure(
        figsize=(FW, FH),
        dpi=(dpi or _PLOT_CFG["dpi"]),
    )

    gs = GridSpec(
        2,
        2,
        width_ratios=(4, 1),
        height_ratios=(1, 4),
        hspace=0.05,
        wspace=0.05,
    )

    ax_top = fig.add_subplot(gs[0, 0])
    ax_joint = fig.add_subplot(gs[1, 0], sharex=ax_top)
    ax_right = fig.add_subplot(gs[1, 1], sharey=ax_joint)

    ax_joint.scatter(
        x,
        y,
        s=point_size,
        alpha=alpha,
        edgecolor="none",
        color=color,
    )

    lo = float(np.nanmin([x.min(), y.min()]))
    hi = float(np.nanmax([x.max(), y.max()]))

    pad = 0.05 * (hi - lo if hi > lo else 1.0)

    lo -= pad
    hi += pad

    ax_joint.set_xlim(lo, hi)
    ax_joint.set_ylim(lo, hi)

    if show_identity:
        ax_joint.plot(
            [lo, hi],
            [lo, hi],
            ls="--",
            lw=1.2,
            color="0.65",
            zorder=1,
        )

    if show_regression and n >= 2 and np.std(x) > 0 and np.std(y) > 0:
        slope, intercept = np.polyfit(x, y, 1)

        ax_joint.plot(
            [lo, hi],
            slope * np.array([lo, hi]) + intercept,
            color="black",
            lw=1.5,
            zorder=2,
        )

    format_axis(
        ax_joint,
        xlabel=xlabel,
        ylabel=ylabel,
        compact_ticks=(),
    )

    if title:
        ax_joint.set_title(title)

    if annotate and n >= 2 and np.std(x) > 0 and np.std(y) > 0:
        rp, _ = safe_pearsonr(x, y)
        rs, _ = safe_spearmanr(x, y)

        txt = rf"$r_p = {rp:.2f}$"

        if annotate_spearman:
            txt += "\n" + rf"$r_s = {rs:.2f}$"

        txt += f"\n$n = {n}$"

        ax_joint.text(
            0.04,
            0.96,
            txt,
            transform=ax_joint.transAxes,
            ha="left",
            va="top",
            fontsize=annotate_fontsize,
        )

    gx, dx = _kde_1d(x, lo, hi)
    gy, dy = _kde_1d(y, lo, hi)

    ax_top.plot(gx, dx, lw=2, color=color)
    ax_top.axis("off")

    ax_right.plot(dy, gy, lw=2, color=color)
    ax_right.axis("off")

    plt.tight_layout()

    return fig, (ax_joint, ax_top, ax_right)


set_plot_style()

# ============================================
# Plot helper functions
# ============================================

def species_palette():
    return [
        SPECIES_INFO["AT21"]["color"],
        SPECIES_INFO["NB21"]["color"],
        SPECIES_INFO["OS21"]["color"],
    ]


def outside_legend(
    ax,
    *,
    title="Species",
    n_items=3,
    x=1.02,
    y=1.00,
):
    handles, labels = ax.get_legend_handles_labels()

    ax.legend(
        handles[:n_items],
        labels[:n_items],
        title=title,
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(x, y),
        borderaxespad=0,
    )

    return ax

In [ ]:
# ============================================
# Load candidate-analysis inputs
# ============================================

RESULT_DIR = Path(
    "/mnt/d/Ibnu/Programming/data/regression/results/lgbm"
)

SPECIES = ["AT21", "NB21", "OS21"]


# --------------------------------------------
# Candidate table
# --------------------------------------------

candidate_file = (
    RESULT_DIR
    / "lgbm.species_specific_candidate_table.importance_p90.sss_p90.tsv.gz"
)

candidate_all_df = pd.read_csv(
    candidate_file,
    sep="\t",
)

candidate_selected_df = (
    candidate_all_df[
        candidate_all_df["Selected"]
    ]
    .copy()
    .reset_index(drop=True)
)

candidate_features = sorted(
    candidate_selected_df["Feature"].unique()
)

feature_metadata_df = (
    candidate_selected_df[
        ["Feature", "Region", "Feature_type"]
    ]
    .drop_duplicates()
    .sort_values("Feature")
    .reset_index(drop=True)
)


# --------------------------------------------
# Load reusable LightGBM data bundles
# --------------------------------------------

lgbm_data = {}

for species in SPECIES:

    bundle_file = (
        RESULT_DIR
        / f"{species}.lgbm.data_bundle.pkl.gz"
    )

    with gzip.open(bundle_file, "rb") as f:
        bundle = pickle.load(f)

    assert bundle["species"] == species

    assert bundle["feature_names"] == list(
        bundle["x_train_raw"].columns
    )

    assert bundle["feature_names"] == list(
        bundle["x_test_raw"].columns
    )

    assert bundle["feature_names"] == list(
        bundle["x_train_model"].columns
    )

    assert bundle["feature_names"] == list(
        bundle["x_test_model"].columns
    )

    missing_features = (
        set(candidate_features)
        - set(bundle["feature_names"])
    )

    assert len(missing_features) == 0, (
        f"{species}: missing candidate features: "
        f"{sorted(missing_features)}"
    )

    lgbm_data[species] = bundle


# --------------------------------------------
# Input summary
# --------------------------------------------

print("Candidate-analysis inputs")
print("-------------------------")
print(f"Candidate rows    : {len(candidate_selected_df)}")
print(f"Unique features   : {len(candidate_features)}")
print()

for species in SPECIES:

    bundle = lgbm_data[species]

    print(
        species,
        "| train:",
        bundle["x_train_raw"].shape,
        "| test:",
        bundle["x_test_raw"].shape,
    )

print()
print("Feature metadata:")
display(feature_metadata_df.head(10))

Candidate-analysis inputs
-------------------------
Candidate rows    : 70
Unique features   : 61

AT21 | train: (5698, 464) | test: (1428, 464)
NB21 | train: (4305, 464) | test: (1076, 464)
OS21 | train: (4445, 463) | test: (1107, 463)

Feature metadata:


,Feature,Region,Feature_type
0,3'UTR.ACA-freq,3'UTR,Nucleotide_kmer_freq
1,3'UTR.AG-freq,3'UTR,Nucleotide_kmer_freq
2,3'UTR.AUC-freq,3'UTR,Nucleotide_kmer_freq
3,3'UTR.G-freq,3'UTR,Nucleotide_kmer_freq
4,3'UTR.GAU-freq,3'UTR,Nucleotide_kmer_freq
5,3'UTR.GUA-freq,3'UTR,Nucleotide_kmer_freq
6,3'UTR.Length,3'UTR,Length
7,3'UTR.UAG-freq,3'UTR,Nucleotide_kmer_freq
8,3'UTR.UC-freq,3'UTR,Nucleotide_kmer_freq
9,5'UTR.ACA-freq,5'UTR,Nucleotide_kmer_freq


In [ ]:
# ============================================
# Build master candidate-feature dataset
# Raw biological feature values only
# ============================================

master_df_list = []

for species in SPECIES:

    bundle = lgbm_data[species]

    # ----------------------------------------
    # Raw feature values
    # ----------------------------------------

    x_raw = pd.concat(
        [
            bundle["x_train_raw"],
            bundle["x_test_raw"],
        ],
        axis=0,
    )

    x_raw = x_raw.loc[:, candidate_features].copy()

    # ----------------------------------------
    # Metadata
    # ----------------------------------------

    metadata = pd.concat(
        [
            bundle["train_metadata"],
            bundle["test_metadata"],
        ],
        axis=0,
    )

    # ----------------------------------------
    # Integrity checks
    # ----------------------------------------

    assert x_raw.index.equals(metadata.index)

    # ----------------------------------------
    # Combine
    # ----------------------------------------

    df = pd.concat(
        [
            metadata,
            x_raw,
        ],
        axis=1,
    )

    df.insert(
        0,
        "Species",
        SPECIES_INFO[species]["label"],
    )

    df.insert(
        1,
        "Species_key",
        species,
    )

    master_df_list.append(df)

# ============================================
# Combine all species
# ============================================

candidate_master_df = pd.concat(
    master_df_list,
    ignore_index=True,
)

print("Master candidate dataset")
print("------------------------")
print("Rows      :", len(candidate_master_df))
print("Columns   :", candidate_master_df.shape[1])
print("Features  :", len(candidate_features))
print()

display(
    candidate_master_df.head()
)

display(
    candidate_master_df["Species"]
    .value_counts()
)

Master candidate dataset
------------------------
Rows      : 18059
Columns   : 70
Features  : 61



,Species,Species_key,var_id,trans_id,gene_id,dataset,observed_raw,observed_model,predicted_model,3'UTR.ACA-freq,...,mRNA.GCU-freq,mRNA.GU-freq,mRNA.GUA-freq,mRNA.K-freq,mRNA.Length,mRNA.MFE,mRNA.U-freq,mRNA.UAG-freq,mRNA.UC-freq,mRNA.Y-freq
0,AT,AT21,AT5G05370.1.1591901.1590815,AT5G05370.1,AT5G05370,Training,0.632897,0.054841,0.418203,6.172840,...,9.661836,55.555556,9.661836,545.893720,414,-149.789993,323.671498,2.415459,91.787440,509.661836
1,AT,AT21,AT5G16060.1.5246121.5247413,AT5G16060.1,AT5G16060,Training,0.365525,-1.094430,-0.499050,9.803922,...,15.065913,60.263653,20.715631,508.474576,531,-176.679993,280.602637,11.299435,67.796610,455.743879
2,AT,AT21,AT2G34160.1.14426246.14427367,AT2G34160.1,AT2G34160,Training,0.600538,-0.073162,-0.132563,13.605442,...,17.361111,48.611111,5.208333,529.513889,576,-211.080002,300.347222,10.416667,36.458333,451.388889
3,AT,AT21,AT5G54600.1.22183004.22184509,AT5G54600.1,AT5G54600,Training,0.544256,-0.302488,-0.542934,0.000000,...,12.096774,49.731183,13.440860,465.053763,744,-262.779999,255.376344,5.376344,77.956989,473.118280
4,AT,AT21,AT2G23340.1.9937988.9938873,AT2G23340.1,AT2G23340,Training,1.047809,1.491571,0.360980,19.108280,...,11.299435,66.666667,9.039548,562.711864,885,-370.059998,279.096045,6.779661,56.497175,454.237288


Species
AT    7126
OS    5552
NB    5381
Name: count, dtype: int64

In [ ]:
# ============================================
# Candidate feature quality assessment
# ============================================

summary_rows = []

for feature in candidate_features:

    meta = feature_metadata_df.loc[
        feature_metadata_df["Feature"] == feature
    ].iloc[0]

    values = candidate_master_df[feature]

    summary_rows.append({

        "Feature": feature,
        "Region": meta["Region"],
        "Feature_type": meta["Feature_type"],

        "n": values.notna().sum(),
        "missing_n": values.isna().sum(),
        "missing_pct": values.isna().mean() * 100,

        "zero_n": (values == 0).sum(),
        "zero_pct": (values == 0).mean() * 100,

        "mean": values.mean(),
        "std": values.std(),

        "median": values.median(),

        "q25": values.quantile(0.25),
        "q75": values.quantile(0.75),

        "IQR": (
            values.quantile(0.75)
            - values.quantile(0.25)
        ),

        "min": values.min(),
        "max": values.max(),

        "skew": stats.skew(
            values,
            nan_policy="omit",
        ),

        "kurtosis": stats.kurtosis(
            values,
            nan_policy="omit",
        ),

        "variance": values.var(),

        "cv": (
            values.std()
            / values.mean()
            if values.mean() != 0
            else np.nan
        ),

    })

candidate_feature_summary_df = (
    pd.DataFrame(summary_rows)
    .sort_values(
        [
            "Region",
            "Feature_type",
            "Feature",
        ]
    )
    .reset_index(drop=True)
)

print("Candidate feature summary")
print("-------------------------")
print(candidate_feature_summary_df.shape)

display(candidate_feature_summary_df.head(20))

Candidate feature summary
-------------------------
(61, 20)


,Feature,Region,Feature_type,n,missing_n,missing_pct,zero_n,zero_pct,mean,std,median,q25,q75,IQR,min,max,skew,kurtosis,variance,cv
0,3'UTR.Length,3'UTR,Length,18059,0,0.0,0,0.000000,186.203943,61.891461,179.000000,144.000000,223.000000,79.000000,22.00000,508.000000,0.687569,0.999343,3830.552989,0.332385
1,3'UTR.ACA-freq,3'UTR,Nucleotide_kmer_freq,18059,0,0.0,3377,18.699817,10.906670,9.197048,9.433962,4.716981,15.789474,11.072493,0.00000,104.166667,1.264204,3.037482,84.585688,0.843250
2,3'UTR.AG-freq,3'UTR,Nucleotide_kmer_freq,18059,0,0.0,84,0.465142,46.507903,17.546186,45.918367,34.482759,57.915058,23.432299,0.00000,148.148148,0.262620,0.388465,307.868652,0.377273
3,3'UTR.AUC-freq,3'UTR,Nucleotide_kmer_freq,18059,0,0.0,1274,7.054654,17.244439,11.217992,15.625000,9.302326,23.529412,14.227086,0.00000,105.263158,0.978087,2.059550,125.843351,0.650528
4,3'UTR.G-freq,3'UTR,Nucleotide_kmer_freq,18059,0,0.0,0,0.000000,188.797438,39.073296,188.172043,163.346614,212.962963,49.616349,21.73913,374.233129,0.199195,0.876461,1526.722427,0.206959
5,3'UTR.GAU-freq,3'UTR,Nucleotide_kmer_freq,18059,0,0.0,978,5.415582,19.186733,11.363942,18.181818,11.152416,25.773196,14.620780,0.00000,128.205128,0.828671,2.091589,129.139181,0.592281
6,3'UTR.GUA-freq,3'UTR,Nucleotide_kmer_freq,18059,0,0.0,1437,7.957251,16.162598,10.243728,15.384615,8.771930,22.123894,13.351964,0.00000,93.896714,0.861283,2.049288,104.933957,0.633792
7,3'UTR.UAG-freq,3'UTR,Nucleotide_kmer_freq,18059,0,0.0,2818,15.604408,11.565705,8.534128,10.752688,5.586592,16.611296,11.024704,0.00000,59.701493,0.774184,0.797466,72.831340,0.737882
8,3'UTR.UC-freq,3'UTR,Nucleotide_kmer_freq,18059,0,0.0,16,0.088598,64.470827,22.406778,62.857143,49.382716,77.419355,28.036639,0.00000,204.081633,0.587652,1.236588,502.063696,0.347549
9,5'UTR.Length,5'UTR,Length,18059,0,0.0,0,0.000000,66.484911,36.191666,63.000000,43.000000,83.000000,40.000000,1.00000,339.000000,1.423299,4.442914,1309.836673,0.544359


In [ ]:
# ============================================
# Raw vs transformed feature distributions
# ============================================

transform_summary = []

for species_key in SPECIES:

    species = SPECIES_INFO[species_key]["label"]
    bundle = lgbm_data[species_key]

    raw_df = candidate_raw_wide_df[
        candidate_raw_wide_df["Species"] == species
    ][candidate_features]

    model_df = pd.concat(
        [
            bundle["x_train_model"][candidate_features],
            bundle["x_test_model"][candidate_features],
        ],
        axis=0,
    )

    for feature in candidate_features:

        raw = raw_df[feature].dropna()

        model = model_df[feature].dropna()

        transform_summary.append({

            "Species": species,
            "Feature": feature,

            "Raw_mean": raw.mean(),
            "Raw_std": raw.std(),
            "Raw_skew": stats.skew(raw),
            "Raw_kurtosis": stats.kurtosis(raw),

            "Model_mean": model.mean(),
            "Model_std": model.std(),
            "Model_skew": stats.skew(model),
            "Model_kurtosis": stats.kurtosis(model),

            "Abs_skew_reduction":
                abs(stats.skew(raw))
                - abs(stats.skew(model)),

            "Abs_kurtosis_reduction":
                abs(stats.kurtosis(raw))
                - abs(stats.kurtosis(model)),
        })

transform_summary_df = pd.DataFrame(transform_summary)

print()
print("Transformation summary")
print("----------------------")

display(transform_summary_df.head())

print()

display(
    transform_summary_df[
        [
            "Raw_skew",
            "Model_skew",
            "Abs_skew_reduction",
            "Raw_kurtosis",
            "Model_kurtosis",
            "Abs_kurtosis_reduction",
        ]
    ].describe()
)


Transformation summary
----------------------


,Species,Feature,Raw_mean,Raw_std,Raw_skew,Raw_kurtosis,Model_mean,Model_std,Model_skew,Model_kurtosis,Abs_skew_reduction,Abs_kurtosis_reduction
0,AT,3'UTR.ACA-freq,11.284752,9.932164,1.298475,2.488143,0.007350,1.002856,-0.140756,-0.562721,1.157719,1.925421
1,AT,3'UTR.AG-freq,43.455944,18.092570,0.533762,1.058897,0.001752,0.995550,0.040986,0.587456,0.492776,0.471442
2,AT,3'UTR.AUC-freq,18.525699,11.346800,0.841680,1.544945,0.001476,0.997725,-0.021338,0.206461,0.820342,1.338484
3,AT,3'UTR.G-freq,172.249670,34.196736,-0.021329,1.015588,-0.006338,0.998494,0.058814,1.018235,-0.037485,-0.002647
4,AT,3'UTR.GAU-freq,19.107572,11.747014,1.073421,3.768234,-0.005635,0.993710,-0.011830,0.650478,1.061591,3.117757


,Raw_skew,Model_skew,Abs_skew_reduction,Raw_kurtosis,Model_kurtosis,Abs_kurtosis_reduction
count,183.000000,183.000000,183.000000,183.000000,183.000000,183.000000
mean,0.987779,0.047588,0.966715,3.634608,-0.168193,2.975440
std,1.080766,0.322689,0.776834,5.784035,0.875991,5.568702
min,-2.616953,-0.387579,-0.210146,-0.516995,-1.853464,-0.761201
25%,0.362704,-0.048799,0.351416,0.554500,-0.694940,0.091173
50%,0.819391,0.009006,0.790581,1.511953,-0.055550,0.900703
75%,1.511256,0.043788,1.436165,3.938444,0.431036,3.089335
max,4.897579,1.886963,3.610468,34.854188,2.614251,33.488241
